# StyleGAN2-ADA — Hernia test run

**Before running:**
1. Set runtime to GPU: Runtime → Change runtime type → **T4 GPU**
2. Upload `stylegan_dataset_hernia_256px.zip` to your **Google Drive** (anywhere, e.g. root)
3. Run cells top to bottom

In [ ]:
# Cell 1 — verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU: ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2 — mount Google Drive and point to the zip
from google.colab import drive
drive.mount('/content/drive')

# Update this path if you put the zip in a subfolder
DATASET_ZIP = '/content/drive/MyDrive/stylegan_dataset_hernia_256px.zip'

import os
assert os.path.exists(DATASET_ZIP), f'Not found: {DATASET_ZIP}'

In [ ]:
# Cell 3 — clone StyleGAN2-ADA and install deps
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch
%pip install -q click requests tqdm pyspng ninja imageio-ffmpeg==0.4.3

In [ ]:
# Cell 4 — patch Python 3.12 / PyTorch 2.x incompatibility
# Sampler.__init__ no longer accepts a data_source argument
!sed -i 's/super().__init__(dataset)/super().__init__()/' \
    stylegan2-ada-pytorch/torch_utils/misc.py

# Verify the fix
!grep -n 'super().__init__' stylegan2-ada-pytorch/torch_utils/misc.py

In [ ]:
# Cell 5 — train
# 50 kimg on a T4 at 256px ≈ 30-60 min
# snap=2 saves a checkpoint every 2 kimg
KIMG   = 50
OUTDIR = '/content/stylegan_output_hernia'

!python stylegan2-ada-pytorch/train.py \
    --outdir={OUTDIR} \
    --data={DATASET_ZIP} \
    --gpus=1 \
    --cfg=paper256 \
    --kimg={KIMG} \
    --mirror=0 \
    --aug=ada \
    --target=0.6 \
    --snap=2 \
    --metrics=none

In [ ]:
# Cell 6 — generate 16 samples from the latest checkpoint
import glob, os

pkls = sorted(glob.glob(f'{OUTDIR}/**/*.pkl', recursive=True))
if not pkls:
    print('No .pkl found. Contents of', OUTDIR, ':')
    for root, dirs, files in os.walk(OUTDIR):
        for f in files:
            print(' ', os.path.join(root, f))
else:
    latest = pkls[-1]
    print('Using checkpoint:', latest)

    !python stylegan2-ada-pytorch/generate.py \
        --outdir=/content/samples \
        --trunc=0.7 \
        --seeds=0-15 \
        --network={latest}

In [ ]:
# Cell 7 — display samples inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

imgs = sorted(glob.glob('/content/samples/*.png'))
if not imgs:
    print('No samples yet — did Cell 6 complete successfully?')
else:
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, path in zip(axes.flat, imgs):
        ax.imshow(mpimg.imread(path), cmap='gray')
        ax.axis('off')
    plt.suptitle(f'StyleGAN2-ADA Hernia @ {KIMG} kimg', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 8 — save checkpoint + samples to Drive
import shutil, glob, os
dest = '/content/drive/MyDrive/stylegan_hernia_results'
os.makedirs(dest, exist_ok=True)
pkls = sorted(glob.glob(f'{OUTDIR}/**/*.pkl', recursive=True))
if pkls:
    shutil.copy(pkls[-1], dest)
    print('Checkpoint saved:', pkls[-1])
if os.path.exists('/content/samples'):
    shutil.copytree('/content/samples', f'{dest}/samples', dirs_exist_ok=True)
    print('Samples saved to Drive:', dest)